# DBSCAN

## Overview

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) groups points that are closely packed in high-density regions and marks points in low-density regions as noise (-1).

**Key advantages over k-means:**
- Does not require k to be specified
- Finds arbitrarily shaped clusters
- Explicitly identifies outliers as noise
- Robust to varying cluster densities (with HDBSCAN)

**Two hyperparameters:**

| Parameter | Meaning | Effect |
|---|---|---|
| `eps` | Neighbourhood radius | Larger -> fewer, larger clusters |
| `min_samples` | Min points to form a core point | Larger -> more noise points |

HDBSCAN (hierarchical DBSCAN) eliminates the need to set `eps` and handles varying densities — prefer it for most practical work.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(42)
# Create non-spherical clusters + noise
t = np.linspace(0, 2*np.pi, 200)
cluster1 = np.column_stack([3*np.cos(t), np.sin(t)]) + rng.normal(0, 0.15, (200,2))
cluster2 = np.column_stack([3*np.cos(t)+5, 2*np.sin(t)+2]) + rng.normal(0, 0.15, (200,2))
noise = rng.uniform(-1, 8, (30, 2))
X = np.vstack([cluster1, cluster2, noise])
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
print(f"Dataset: {X.shape} (includes 30 noise points)")

---
## Choosing eps with the k-Distance Plot

In [ ]:
# Plot sorted distances to the min_samples-th nearest neighbour
# Elbow in this plot suggests a good eps
min_samples = 5
nn = NearestNeighbors(n_neighbors=min_samples)
nn.fit(X_sc)
dists, _ = nn.kneighbors(X_sc)
kdist = np.sort(dists[:, -1])[::-1]
plt.figure(figsize=(7,4))
plt.plot(kdist, color="steelblue", lw=1.5)
plt.xlabel("Points sorted by distance"); plt.ylabel(f"{min_samples}-NN distance")
plt.title("k-Distance Plot: Choose eps at the elbow")
plt.tight_layout(); plt.show()
print("Look for the elbow in this plot to choose eps")

---
## Fitting DBSCAN

In [ ]:
db = DBSCAN(eps=0.35, min_samples=5)
labels = db.fit_predict(X_sc)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()
print(f"Clusters found: {n_clusters}")
print(f"Noise points:   {n_noise} ({n_noise/len(labels):.1%})")
if n_clusters > 1:
    mask = labels != -1
    print(f"Silhouette (non-noise): {silhouette_score(X_sc[mask], labels[mask]):.3f}")
fig, ax = plt.subplots(figsize=(7,5))
unique_labels = sorted(set(labels))
colors = plt.cm.tab10(np.linspace(0, 0.4, max(n_clusters,1)))
for k, col in zip([l for l in unique_labels if l != -1], colors):
    ax.scatter(X_sc[labels==k,0], X_sc[labels==k,1], s=20, alpha=0.7, color=col, label=f"C{k}")
if -1 in labels:
    ax.scatter(X_sc[labels==-1,0], X_sc[labels==-1,1], s=30, c="black",
               marker="x", label="Noise", alpha=0.8)
ax.set_title(f"DBSCAN: {n_clusters} clusters, {n_noise} noise points")
ax.legend(); plt.tight_layout(); plt.show()

---
## eps Sensitivity

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(13,4))
for ax, eps_val in zip(axes, [0.15, 0.35, 0.8]):
    lbl = DBSCAN(eps=eps_val, min_samples=5).fit_predict(X_sc)
    nc = len(set(lbl)) - (1 if -1 in lbl else 0)
    nn_ = (lbl==-1).sum()
    for k in sorted(set(lbl)):
        color = "black" if k == -1 else None
        marker = "x" if k == -1 else "o"
        ax.scatter(X_sc[lbl==k,0], X_sc[lbl==k,1], s=15, alpha=0.7,
                   c=color, marker=marker)
    ax.set_title(f"eps={eps_val}: {nc} clusters, {nn_} noise")
plt.suptitle("DBSCAN Sensitivity to eps (min_samples=5)")
plt.tight_layout(); plt.show()

---
## HDBSCAN (Hierarchical DBSCAN)

In [ ]:
try:
    import hdbscan
    hdb = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5)
    labels_h = hdb.fit_predict(X_sc)
    nc_h = len(set(labels_h)) - (1 if -1 in labels_h else 0)
    nn_h = (labels_h==-1).sum()
    print(f"HDBSCAN: {nc_h} clusters, {nn_h} noise points")
    print("HDBSCAN advantages: no eps parameter, handles varying density")
except ImportError:
    print("hdbscan not installed: pip install hdbscan")
    print("HDBSCAN is generally preferred over DBSCAN for:")
    print("  - Varying cluster densities")
    print("  - Avoiding eps grid search")
    print("  - Soft cluster membership probabilities")

---

## Common Pitfalls

**1. Choosing eps without the k-distance plot**  
DBSCAN is highly sensitive to eps. Too small: almost everything is noise. Too large: all points merge into one cluster. Always plot the sorted k-NN distances and look for the elbow before selecting eps.

**2. Forgetting that noise points (label=-1) are excluded from silhouette computation**  
`silhouette_score` requires at least 2 labels, and noise points distort silhouette values. Always filter `labels != -1` before computing silhouette for DBSCAN results.

**3. Using DBSCAN when cluster densities vary substantially**  
DBSCAN uses a single global eps, so it cannot find clusters that have different densities. A dense cluster and a sparse cluster require different eps values. Use HDBSCAN for density-varying data.

**4. Standardising without fitting the scaler on training data**  
As with all distance-based methods, always fit `StandardScaler` on the data being clustered (or training data in a pipeline) and apply `transform` consistently. Do not refit the scaler when predicting cluster membership for new points.

**5. Treating DBSCAN noise points as anomalies without investigation**  
DBSCAN noise points are observations that do not belong to any dense region — this can indicate true outliers, sparse but valid observations, or simply that eps is too small. Always inspect noise points before deciding how to handle them.
---
*python_methods_library - Samantha McGarrigle*